In [1]:
from utils import createSparkSession, getThroughputRate, write_postgres_table

In [2]:
spark, stagemetrics = createSparkSession('P1M1')

26/11/04 06:58:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## M0-2 Stage as text

In [ ]:
stagemetrics.begin()

In [ ]:
aisles_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/aisles.csv", header=True)
)
departments_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/departments.csv", header=True)
)
order_products__prior_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/order_products__prior.csv", header=True)
)
order_products__train_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/order_products__train.csv", header=True)
)
orders_df = (
    spark.read
        .option('columnNameOfCorruptRecord', 'bad_record')
        .csv("s3a://raw/InstaCart/orders.csv", header=True)
)
products_df = (
    spark.read
        .option('columnNameOfCorruptRecord', 'bad_record')
        .csv("s3a://raw/InstaCart/products.csv", header=True)
)

In [ ]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [ ]:
write_postgres_table(aisles_df, 'staging_text.aisles')
write_postgres_table(departments_df, 'staging_text.departments')
write_postgres_table(order_products_df, 'staging_text.order_products')
write_postgres_table(orders_df, 'staging_text.orders')
write_postgres_table(products_df, 'staging_text.products')

In [ ]:
stagemetrics.end()

In [ ]:
stagemetrics.print_report()
stagemetrics.print_memory_report()
getThroughputRate(stagemetrics.aggregate_stagemetrics())

## M0-3 Parsing fixes

In [3]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ByteType, ShortType

In [4]:
aisles_schema = StructType([
    StructField('aisle_id', IntegerType(), False),
    StructField('aisle', StringType(), True),
    StructField('bad_record', StringType(), True),
])

departments_schema = StructType([
    StructField('department_id', IntegerType(), False),
    StructField('department', StringType(), True),
    StructField('bad_record', StringType(), True),
])

order_products_schema = StructType([
    StructField('product_id', IntegerType(), False),
    StructField('order_id', IntegerType(), False),
    StructField('add_to_cart_order', IntegerType(), True),
    StructField('reordered', ByteType(), True),
    StructField('bad_record', StringType(), True),
])

products_schema = StructType([
    StructField('product_id', IntegerType(), False),
    StructField('product_name', StringType(), True),
    StructField('aisle_id', IntegerType(), True),
    StructField('department_id', IntegerType(), True),
    StructField('bad_record', StringType(), True),
])

orders_schema = StructType([
    StructField('order_id', IntegerType(), False),
    StructField('user_id', IntegerType(), False),
    StructField('eval_set', StringType(), True),
    StructField('order_number', IntegerType(), True),
    StructField('order_dow', IntegerType(), True),
    StructField('order_hour_of_day', IntegerType(), True),
    StructField('days_since_prior_order', DoubleType(), True),
    StructField('bad_record', StringType(), True),
])

In [5]:
stagemetrics.begin()

In [6]:
aisles_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/aisles.csv", header=True, schema=aisles_schema)
)
departments_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/departments.csv", header=True, schema=departments_schema)
)
order_products__prior_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/order_products__prior.csv", header=True, schema=order_products_schema)
)
order_products__train_df = (
    spark.read
    .option('columnNameOfCorruptRecord', 'bad_record')
    .csv("s3a://raw/InstaCart/order_products__train.csv", header=True, schema=order_products_schema)
)
orders_df = (
    spark.read
        .option('columnNameOfCorruptRecord', 'bad_record')
        .csv("s3a://raw/InstaCart/orders.csv", header=True, schema=orders_schema)
)
products_df = (
    spark.read
        .option('columnNameOfCorruptRecord', 'bad_record')
        .csv("s3a://raw/InstaCart/products.csv", header=True, schema=products_schema)
)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


1. No bad records or parsing issees found with inferSchema = True.
2. With enforced schema: 

In [14]:
aisles_df.filter('bad_record is not NULL').show()

[Stage 0:>                                                          (0 + 1) / 1]

+--------+-----+----------+
|aisle_id|aisle|bad_record|
+--------+-----+----------+
+--------+-----+----------+



In [15]:
departments_df.filter('bad_record is not NULL').show()

+-------------+----------+----------+
|department_id|department|bad_record|
+-------------+----------+----------+
+-------------+----------+----------+



In [16]:
order_products__prior_df.filter('bad_record is not NULL').show()

[Stage 3:============================================>              (3 + 1) / 4]

+----------+--------+-----------------+---------+----------+
|product_id|order_id|add_to_cart_order|reordered|bad_record|
+----------+--------+-----------------+---------+----------+
+----------+--------+-----------------+---------+----------+



In [17]:
order_products__train_df.filter('bad_record is not NULL').show()

+----------+--------+-----------------+---------+----------+
|product_id|order_id|add_to_cart_order|reordered|bad_record|
+----------+--------+-----------------+---------+----------+
+----------+--------+-----------------+---------+----------+



In [18]:
products_df.filter('bad_record is not NULL').show()

+----------+--------------------+--------+-------------+--------------------+
|product_id|        product_name|aisle_id|department_id|          bad_record|
+----------+--------------------+--------+-------------+--------------------+
|      6816|"Scotch Kids 5"" ...|    NULL|         NULL|6816,"Scotch Kids...|
+----------+--------------------+--------+-------------+--------------------+



In [19]:
orders_df.filter('bad_record is not NULL').show()

[Stage 8:>                                                          (0 + 2) / 2]

+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
|order_id|user_id|eval_set|order_number|order_dow|order_hour_of_day|days_since_prior_order|bad_record|
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+
+--------+-------+--------+------------+---------+-----------------+----------------------+----------+



In [ ]:
order_products_df = order_products__prior_df.union(order_products__train_df)

In [20]:
write_postgres_table(aisles_df, 'staging_text.aisles')
write_postgres_table(departments_df, 'staging_text.departments')
write_postgres_table(order_products_df, 'staging_text.order_products')
write_postgres_table(orders_df, 'staging_text.orders')
write_postgres_table(products_df, 'staging_text.products')

In [8]:
from data_profiling import ProfileReport

In [9]:
ProfileReport(aisles_df).to_file('/home/ubuntu/work_dir/profiling/aisles.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:38 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:14:41 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [10]:
ProfileReport(departments_df).to_file('/home/ubuntu/work_dir/profiling/departments.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:14:53 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:14:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [11]:
ProfileReport(order_products__prior_df).to_file('/home/ubuntu/work_dir/profiling/order_products__prior.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:16:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:16:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:04 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:17:17 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
ProfileReport(order_products__train_df).to_file('/home/ubuntu/work_dir/profiling/order_products__train.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:18:37 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:18:41 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:45 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:18:47 WARN CacheManager: Asked to cache already cached data.
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
ProfileReport(orders_df).to_file('/home/ubuntu/work_dir/profiling/orders.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:05 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:15 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:20 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:23 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:27 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:30 WARN CacheManager: Asked to cache already cached data.        
26/10/31 11:19:34 WARN CacheManager: Asked to cache already cached data.        
                                                                                

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [14]:
ProfileReport(products_df).to_file('/home/ubuntu/work_dir/profiling/products.html')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

26/10/31 11:19:49 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:50 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:52 WARN CacheManager: Asked to cache already cached data.
26/10/31 11:19:54 WARN CacheManager: Asked to cache already cached data.


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [13]:
spark.stop()